# Method B — essential 1× seed-42 check and handoff

For Suraj Shinde. Open this notebook on **Sol with any Python 3.10+ kernel**.
Use **Run → Run All Cells**, or Shift+Enter through the cells in order.

**Estimated time: 5–20 minutes**, allowing longer for a large archive or slow scratch storage.
No GPU training or generation is launched. No package installation is needed.
This notebook checks your **four already-completed primary B runs**:

| Model | Parent seeds | B training seed | Additional token budget |
|---|---|---|---|
| Qwen 0.5B | 17 and 123 | 42 | 1× |
| Qwen 1.5B | 17 and 123 | 42 | 1× |

**B means fresh stratified random selection**, as in Section 7 of the main design.
It continues from each R1000 parent using 1,000 fresh examples under source, length,
and supervised-token controls. The appended Task 2 uses different method labels.

The notebook verifies saved evidence, independently recomputes scores from saved correctness
labels, and creates a handoff with final adapters, predictions, selection records, and code.
It does not reload models, regenerate predictions, regrade answers, or re-encode training labels.

**Bounded scope:** it completes this seed-42 development handoff, not the full final study.
It leaves additional budgets, new training repetitions, Numina transfer, and final-test
evaluation deferred. Previously completed seed-137/2026 results stay in your original project.
The main document targets three training seeds for final main comparisons.

Stop the old overnight worker before running this notebook. An active worker lock causes
a clear stop; no process is killed. If a check fails, upload the generated
`ESSENTIAL_CHECK.json` for diagnosis. Training does not start automatically.


## 1. Load the checker

This cell defines the functions. It does not start any work.


In [ ]:
"""CPU-only evidence check and handoff for existing Method B 1x, training seed 42."""
import csv
import fcntl
import hashlib
import importlib.util
import io
import json
import math
import random
import tarfile
import time
import uuid
from collections import Counter
from contextlib import ExitStack, contextmanager
from datetime import datetime, timezone
from pathlib import Path

STUDY = 'B_method_B_document_v3'
PROFILES = [(m, p) for m in ('0.5B', '1.5B') for p in (17, 123)]
EXPECTED_SELECT_SHA = '5c9241231efc449fccbe464ce3198b87af20de8041aa699149f9c702f83bf7de'
COMPARE_FIELDS = ('model', 'revision', 'precision', 'max_new_tokens', 'batch_size',
    'stop_at_question', 'question_text', 'prompt_sha256', 'add_special_tokens',
    'decoding', 'eos_token_id', 'transformers', 'peft', 'torch', 'gpu', 'cuda',
    'parser_version', 'math_verify', 'eval_file_sha256', 'answer_parser_sha256',
    'evaluate_py_sha256')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def rows(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    tmp.replace(path)


@contextmanager
def quiet_project(root):
    # Use the same locks as the previous recovery/overnight runners.
    with ExitStack() as stack:
        for path in (root/'outputs'/STUDY/'overnight/worker.lock', root/'outputs'/STUDY/'evaluation_recovery.lock'):
            path.parent.mkdir(parents=True, exist_ok=True)
            handle = stack.enter_context(path.open('a+'))
            try:
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError('An existing worker holds a lock. Wait for it to finish: ' + str(path))
        yield


def selection_check(root):
    base = root/'outputs'/STUDY
    work = base/'mixed/budget_1x'
    shared = base/'shared'
    source = root/'method_b_document_v3_code/select_b.py'
    require(sha(source) == EXPECTED_SELECT_SHA, 'Selection helper differs from reviewed source')
    spec = importlib.util.spec_from_file_location('essential_selection', source)
    s = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(s)  # Standard library only; does not invoke the solver.
    audit = read(work/'selection_audit.json')
    bank_audit = read(shared/'bank_audit.json')
    parent_path = root/'data/train_n1000.jsonl'
    parent_raw = rows(parent_path)
    parent = s.rebin(parent_raw, s.parent_edges(parent_raw))
    require(parent == rows(shared/'parent_rebinned_reference.jsonl'), 'Parent strata changed')
    parent_hash = sha(parent_path)
    cfg = read(work/'runtime/config.json')
    require(parent_hash == cfg['parent_manifest_sha256'], 'Parent manifest hash differs')
    chosen_path = work/'1x/train_B.jsonl'
    chosen = rows(chosen_path)
    bank_path = shared/'mixed_candidate_bank.jsonl'
    bank = rows(bank_path)
    for data in (parent, bank, chosen):
        s.validate(data)
    require(s.rebin(bank, s.parent_edges(parent_raw)) == bank, 'Fresh-bank strata changed')
    require(len(parent) == len(chosen) == 1000, 'Expected exactly 1000 parent and 1000 fresh rows')
    require(audit.get('passed') is True and sha(chosen_path) == audit['selected_manifest_sha256'],
            'Selection audit failed or selected file changed')
    require(sha(bank_path) == audit['bank_sha256'] == bank_audit['mixed_bank_sha256'], 'Bank hash differs')
    require(parent_hash == bank_audit['parent_manifest_sha256'], 'Bank audit uses another parent')
    require(sha(shared/'parent_length_bins.json') == bank_audit['length_bins_sha256'], 'Saved length-bin file changed')
    require(sha(shared/'clean_development_reference.jsonl') == bank_audit['clean_development_sha256'], 'Reserved development reference changed')
    require(cfg['common_universe_sha256'] == sha(bank_path), 'Runtime bank differs')
    require(cfg['selection_seed'] == 42, 'Selection seed differs')
    original = {r['row_id']: r for r in bank}
    require(all(original.get(r['row_id']) == r for r in chosen), 'Selected row absent from canonical bank')
    seen = {r['problem_group_id'] for r in parent}
    with (shared/'parent_seen_groups.csv').open() as f:
        seen |= {r['problem_group_id'] for r in csv.DictReader(f)}
    require(not seen & {r['problem_group_id'] for r in bank}, 'Fresh bank contains parent-seen groups')
    require(Counter(s.cell(r) for r in chosen) == Counter(s.cell(r) for r in parent), 'Stratum counts differ')
    target, actual = sum(r['supervised_tokens'] for r in parent), sum(r['supervised_tokens'] for r in chosen)
    lo, hi = s.bounds(target, '0.005')
    require(lo <= actual <= hi, 'Global token budget outside +/-0.5%')
    for h in {s.cell(r) for r in parent}:
        ref = sum(r['supervised_tokens'] for r in parent if s.cell(r) == h)
        obs = sum(r['supervised_tokens'] for r in chosen if s.cell(r) == h)
        low, high = s.bounds(ref, '0.05')
        require(low <= obs <= high, 'Stratum token budget differs: ' + str(h))
    priority_path = work/'1x/candidate_priorities.csv'
    with priority_path.open() as f:
        priorities = list(csv.DictReader(f))
    require(len(priorities) == len(bank) and {p['row_id'] for p in priorities} == set(original), 'Priorities incomplete/duplicated')
    selected_ids = {r['row_id'] for r in chosen}
    for p in priorities:
        require(int(p['utility_integer']) == s.utility(p['row_id']) and
                int(p['raw_first32']) == s.hash32('42|random|' + p['row_id']) and
                p['selected'] == str(p['row_id'] in selected_ids), 'Random priority record differs')
    verification = read(work/'label_counts_and_strata_verified.json')
    require(verification.get('passed') is True and verification.get('all_stored_counts_match_actual_labels') is True and
            verification['selection_audit_sha256'] == sha(work/'selection_audit.json'), 'Saved actual-label verification is stale')
    return {'work': work, 'cfg': cfg, 'target_tokens': target, 'actual_tokens': actual,
            'selection_sha256': sha(chosen_path), 'rows': len(chosen)}


def prediction_check(path, frozen, adapter_sha, evaluator_sha):
    data = rows(path)
    cfg = read(path.with_suffix('.config.json'))
    marker = read(path.with_suffix('.validated.json'))
    require(marker['predictions_sha256'] == sha(path) and marker['config_sha256'] == sha(path.with_suffix('.config.json')),
            'Prediction or config changed: ' + str(path))
    signature = marker['signature']
    weight_key = 'run_adapter_sha256' if 'run_adapter_sha256' in signature else 'adapter_sha256'
    require(signature.get(weight_key) == adapter_sha, 'Prediction adapter hash differs: ' + str(path))
    require(signature.get('eval_file_sha256') == sha(frozen) == cfg.get('eval_file_sha256'), 'Evaluation set differs')
    require(signature.get('evaluator_sha256') == evaluator_sha == cfg.get('evaluate_py_sha256'), 'Evaluator code differs')
    ids = [r['problem_id'] for r in data]
    expected = [r['problem_id'] for r in rows(frozen)]
    require(len(expected) == len(set(expected)) == 500, 'Expected 500 unique benchmark questions')
    require(len(ids) == len(set(ids)) == len(expected) and set(ids) == set(expected), 'Incomplete/duplicate prediction IDs: ' + str(path))
    require(all(type(r.get('correct')) is bool for r in data), 'Nonboolean/missing correctness')
    require(all(k in cfg for k in COMPARE_FIELDS), 'Missing evaluator settings')
    return data, cfg


def compare_existing(parent, b, saved, dataset, profile):
    ad, ac = parent
    bd, bc = b
    require(all(ac[k] == bc[k] for k in COMPARE_FIELDS), 'Parent/B evaluator settings differ')
    pa, ba = sum(r['correct'] for r in ad)/len(ad), sum(r['correct'] for r in bd)/len(bd)
    for key, value in [('questions', len(ad)), ('parent_accuracy', pa), ('B_accuracy', ba),
                       ('change_percentage_points', 100*(ba-pa))]:
        require(abs(float(saved[key]) - value) < 1e-8, 'Stored comparison differs: ' + key)
    require(saved['parent_profile'] == profile and saved['training_seed'] == 42 and
            saved['budget_multiplier'] == 1 and saved['dataset'] == dataset, 'Comparison identity differs')
    for key in ('settings_compatible', 'complete_frozen_set_verified', 'real_predictions_only'):
        require(saved.get(key) is True, 'Stored comparison missing validation: ' + key)
    # Recompute fixed-checkpoint paired-item intervals using the original seed/rule.
    a = {r['problem_id']: r['correct'] for r in ad}
    bmap = {r['problem_id']: r['correct'] for r in bd}
    delta = [int(bmap[k])-int(a[k]) for k in sorted(a)]
    rng = random.Random(1729)
    samples = sorted(100*sum(rng.choices(delta, k=len(delta)))/len(delta) for _ in range(2000))
    interval = (samples[int(.025*1999)], samples[math.ceil(.975*1999)])
    require(saved.get('bootstrap_replicates') == 2000, 'Unexpected bootstrap count')
    require(all(abs(float(saved[k])-v) < 1e-8 for k, v in zip(
        ('paired_item_ci95_low_pp', 'paired_item_ci95_high_pp'), interval)), 'Stored item interval differs')
    return dict(saved, setting='mixed', evidence_status='preliminary development comparison',
                development_protection='Parent/evaluator clearance still requires shared verification')


def check_run(root, selection, model, parent_seed):
    work, cfg = selection['work'], selection['cfg']
    key = f'{model}_seed{parent_seed}'
    registry = read(work/'runtime/parent_registry.json')
    entry = registry[key]
    require(entry['parent_manifest_sha256'] == cfg['parent_manifest_sha256'], 'Parent registry manifest differs')
    parent_adapter = root/entry['adapter_relative_path']/'adapter_model.safetensors'
    require(sha(parent_adapter) == entry['expected_adapter_sha256'], 'Original parent adapter changed')
    name = f'B_{model}_p{parent_seed}_t42_1x'
    index_path = work/'run_index.json'
    index = read(index_path) if index_path.is_file() else {}
    run = work/'runs'/index.get(name, name)
    require(run.resolve().is_relative_to((work/'runs').resolve()), 'Run index escapes this condition')
    audit, rc = read(run/'B_completion_audit.json'), read(run/'run_config.json')
    require(audit.get('passed') is True and audit.get('parent_adapter_unchanged') is True and
            audit.get('fresh_optimizer_and_warmup') is True, 'Training completion audit failed')
    require(rc['parent_profile'] == key and rc['continuation_training_seed'] == 42 and
            rc['additional_budget_multiplier'] == 1 and rc['manifest_sha256'] == selection['selection_sha256'], 'Training identity differs')
    require(rc['parent_adapter_sha256'] == entry['expected_adapter_sha256'] and
            rc['model'] == entry['model'] and rc['revision'] == entry['revision'], 'Training lineage differs')
    require(audit['actual_consumed_tokens'] == audit['expected_tokens'] == selection['actual_tokens'] and
            audit['unique_rows'] == selection['rows'], 'One-pass training consumption differs')
    final = run/'final'
    weights = final/'adapter/adapter_model.safetensors'
    record = read(final/'continuation_checkpoint.json')
    require(record.get('kind') == 'B_continuation' and record.get('smoke') is False and
            record['manifest_sha256'] == selection['selection_sha256'] and
            record['parent_adapter_sha256'] == entry['expected_adapter_sha256'], 'Final checkpoint lineage differs')
    require(record['file_sha256']['adapter/adapter_model.safetensors'] == sha(weights), 'Final adapter changed')
    require(record['file_sha256']['run_config.json'] == sha(final/'run_config.json'), 'Final run config changed')
    final_cfg = read(final/'run_config.json')
    for field in ('parent_profile', 'continuation_training_seed', 'manifest_sha256', 'parent_adapter_sha256'):
        require(final_cfg[field] == rc[field], 'Top-level/final checkpoint identity differs')
    smoke_path = work/'smokes'/f'{key}_t42/smoke_report.json'
    smoke = read(smoke_path)
    require(all(smoke.get(k) is True for k in ('passed', 'actual_gpu_execution', 'reload_probe_matched')) and
            smoke['optimizer_updates'] == 4 and smoke['fingerprint']['training_seed'] == 42, 'Saved GPU smoke evidence failed')
    evaluation = work/'evaluation'/run.name
    report = read(evaluation/'B_vs_parent.json')
    require(report.get('final_test_used') is False, 'Unexpected final-test declaration')
    require(len(report['comparisons']) == 2 and {r['dataset'] for r in report['comparisons']} == {'dev_gsm', 'dev_math'}, 'Expected two distinct benchmark comparisons')
    evaluator_sha = sha(work/'runtime/vendor/evaluation/evaluate.py')
    result, files = [], [smoke_path]
    for dataset in ('dev_gsm', 'dev_math'):
        frozen = work/'runtime/vendor/evaluation/data'/f'{dataset}.jsonl'
        bpath = evaluation/f'B_{dataset}.jsonl'
        bdata = prediction_check(bpath, frozen, sha(weights), evaluator_sha)
        require(bdata[1]['model'] == entry['model'] and bdata[1]['revision'] == entry['revision'], 'Evaluation model/revision differs')
        candidates = sorted((root/'outputs'/STUDY/'parent_evaluation_cache').glob(f'{key}_{dataset}_*.jsonl'))
        eligible = []
        for p in candidates:
            try:
                pdata = prediction_check(p, frozen, entry['expected_adapter_sha256'], evaluator_sha)
                if all(pdata[1][k] == bdata[1][k] for k in COMPARE_FIELDS):
                    eligible.append((p, pdata))
            except (OSError, ValueError, KeyError, TypeError):
                pass
        require(len(eligible) == 1, f'Expected one valid compatible parent prediction; found {len(eligible)}: {key}/{dataset}')
        ppath, pdata = eligible[0]
        saved = next(r for r in report['comparisons'] if r['dataset'] == dataset)
        result.append(compare_existing(pdata, bdata, saved, dataset, key))
        files += [frozen, ppath, ppath.with_suffix('.config.json'), ppath.with_suffix('.validated.json')]
    return {'profile': key, 'run': str(run), 'adapter_sha256': sha(weights), 'rows': result,
            'evidence_files': [str(p) for p in files]}


def audit_essential(root, destination):
    root, destination = Path(root).resolve(), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    report = {'scope': 'Method B fresh stratified random; mixed 1x; training seed 42; both sizes/parents',
              'started_utc': datetime.now(timezone.utc).isoformat(), 'passed': False, 'runs': [], 'errors': [],
              'new_training_launched': False, 'new_gpu_evaluation_launched': False,
              'final_test_used': False, 'shared_evaluator_clearance_verified': False}
    try:
        with quiet_project(root):
            selection = selection_check(root)
            report['selection'] = {k: v for k, v in selection.items() if k not in ('work', 'cfg')}
            print('Selection, freshness, random priorities and saved token-label checks passed.', flush=True)
            for model, parent in PROFILES:
                try:
                    item = check_run(root, selection, model, parent)
                    report['runs'].append(item)
                    print('VERIFIED:', item['profile'], 'training seed 42; 2 x 500 predictions', flush=True)
                except (OSError, ValueError, KeyError, TypeError, StopIteration) as exc:
                    report['errors'].append(f'{model}, parent {parent}: {type(exc).__name__}: {exc}')
            report['passed'] = len(report['runs']) == 4 and not report['errors']
    except Exception as exc:
        report['errors'].append(f'{type(exc).__name__}: {exc}')
    write_json(destination/'ESSENTIAL_CHECK.json', report)
    if not report['passed']:
        print('\nEvidence needs repair:\n' + '\n'.join(report['errors']))
        raise RuntimeError('Check stopped; existing results preserved. Upload ' + str(destination/'ESSENTIAL_CHECK.json'))
    write_json(destination/'RESULTS.json', [row for item in report['runs'] for row in item['rows']])
    result_rows = [row for item in report['runs'] for row in item['rows']]
    with (destination/'RESULTS.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(result_rows[0]))
        writer.writeheader()
        writer.writerows(result_rows)
    write_readme(destination, report)
    return report


def write_readme(destination, report):
    lines = ['METHOD B — ESSENTIAL 1x SEED-42 DEVELOPMENT HANDOFF', '',
        'Method: fresh stratified random priorities under source/length/token controls.',
        'Four existing B runs: Qwen 0.5B/1.5B x parent seeds 17/123; B training seed 42.',
        'Every run starts independently from its matching R1000 parent.',
        'Selected data: 1000 fresh examples, one training exposure each.',
        'No new training, generation, package installation, or final-test evaluation was launched.', '',
        'VERIFICATION SCOPE',
        'Checks saved weights, lineage, selections, priorities, counts, token budgets, saved actual-label',
        'verification, complete prediction IDs, settings, file hashes, accuracies and paired-item intervals.',
        'Does not reload models, regenerate predictions, re-encode labels, or independently regrade answers.', '',
        'RESULTS: each row is one 500-question development benchmark comparison.',
        'profile | dataset | parent accuracy | B accuracy | change (percentage points)']
    for item in report['runs']:
        for r in item['rows']:
            lines.append(f"{r['parent_profile']} | {r['dataset']} | {100*r['parent_accuracy']:.2f}% | "
                         f"{100*r['B_accuracy']:.2f}% | {r['change_percentage_points']:+.2f}")
    lines += ['', 'LIMITS AND DEFERRED WORK',
        'Preliminary development evidence. Clean parent/evaluator protection still requires shared verification.',
        'The original training metadata records seven parent-related development groups to disclose.',
        'Seed-42 evidence alone does not establish robustness across training seeds.',
        'Additional completed seed-137/2026 results remain in the original project; they are not included here.',
        'The document targets three seeds for final main comparisons; these repetitions are deferred in this scope.',
        'B-minus-A and C-minus-B require compatible teammate predictions and common evaluation.',
        'The protected final evaluation, Numina transfer, and other token budgets are outside this handoff.',
        '4x was blocked by screened-bank count/token constraints in the previous run.',
        'Negative/inconclusive results must be retained; improvement is not assumed.', '',
        'LOADING AND REPRODUCIBILITY',
        'Each final/adapter directory contains a PEFT adapter for its run_config model/revision.',
        'Use the original compatible model-loading/evaluation code and recorded environment.',
        'Base models, original parent weights, environments, caches, and optimizer checkpoints are excluded.',
        'This is an inference/evidence handoff, not a complete optimizer-resume backup.',
        'Original absolute paths in configurations describe the Sol execution environment.',
        'SHA256.json lists the included file contents; ARCHIVE_RECEIPT.json records the archive hash.', '']
    (Path(destination)/'README.txt').write_text('\n'.join(lines), encoding='utf-8')


def package_essential(root, destination, report):
    root, destination = Path(root).resolve(), Path(destination).resolve()
    require(report.get('passed') is True and len(report.get('runs', [])) == 4, 'Run the verification cell successfully first')
    work = root/'outputs'/STUDY/'mixed/budget_1x'
    shared = root/'outputs'/STUDY/'shared'
    sources = {}
    def add(path):
        path = Path(path)
        if not path.is_file():
            return
        logical = path.absolute()
        require(logical.is_relative_to(root), 'Package source outside the project: ' + str(path))
        sources[str(Path('project')/logical.relative_to(root))] = path
    def tree(directory, exclude=(), suffixes=None):
        for path in sorted(Path(directory).rglob('*')):
            if path.is_file() and not any(p in exclude for p in path.relative_to(directory).parts) and (suffixes is None or path.suffix in suffixes):
                add(path)
    with quiet_project(root):
        # Recheck the complete evidence chain so a stale notebook report cannot package changed results.
        current = audit_essential_unlocked(root, report)
        for path in destination.iterdir():
            if path.suffix in ('.json', '.csv', '.txt') and path.name != 'ARCHIVE_RECEIPT.json':
                sources['handoff/' + path.name] = path
        add(root/'data/train_n1000.jsonl')
        for name in ('bank_audit.json', 'parent_original_reference.jsonl', 'parent_rebinned_reference.jsonl',
                     'parent_length_bins.json', 'parent_seen_groups.csv', 'mixed_candidate_bank.jsonl',
                     'clean_development_reference.jsonl'):
            add(shared/name)
        for name in ('selection_audit.json', 'label_counts_and_strata_verified.json', 'run_index.json'):
            add(work/name)
        tree(work/'1x')
        tree(work/'label_checks')
        tree(work/'training_inputs')
        tree(work/'runtime', exclude=('vendor', 'reference_records', '__pycache__', 'runs'))
        code_suffixes = {'.py', '.json', '.jsonl', '.csv', '.md', '.txt', '.yaml', '.yml', '.toml', '.sh'}
        tree(work/'runtime/vendor', exclude=('__pycache__', '.git', 'data'), suffixes=code_suffixes)
        tree(work/'runtime/reference_records', exclude=('__pycache__',), suffixes=code_suffixes)
        tree(root/'inputs')
        add(root/'Method_B_Essential_Seed42.ipynb')
        for p in sorted((root/'method_b_document_v3_code').glob('*')):
            if p.suffix in ('.py', '.json'):
                add(p)
        for item in current:
            run = Path(item['run'])
            for name in ('run_config.json', 'B_completion_audit.json', 'training_summary.json', 'training_log.jsonl'):
                add(run/name)
            tree(run/'final', exclude=('training_state.pt', '__pycache__'))
            tree(work/'evaluation'/run.name)
            for path in item['evidence_files']:
                add(path)
        manifest = {}
        print('Hashing', len(sources), 'handoff files; adapters and predictions can take a few minutes.', flush=True)
        for i, (name, path) in enumerate(sorted(sources.items()), 1):
            manifest[name] = {'sha256': sha(path), 'bytes': path.stat().st_size}
            if i % 50 == 0:
                print('Hashed', i, '/', len(sources), flush=True)
        write_json(destination/'SHA256.json', manifest)
        sources['handoff/SHA256.json'] = destination/'SHA256.json'
        archive = destination.with_suffix('.tar.gz')
        require(not archive.exists(), 'Archive already exists; use a fresh run folder')
        tmp = archive.with_name(archive.name + '.partial')
        try:
            with tarfile.open(tmp, 'w:gz', compresslevel=1, dereference=True) as tar:
                for i, (name, path) in enumerate(sorted(sources.items()), 1):
                    require(sha(path) == (manifest.get(name) or {'sha256': sha(path)})['sha256'], 'Source changed while packaging: ' + str(path))
                    tar.add(path, arcname=name, recursive=False)
                    if i % 50 == 0:
                        print('Packed', i, '/', len(sources), flush=True)
            tmp.replace(archive)
        except BaseException:
            print('Packaging interrupted. Existing model/data files remain unchanged. Rerun this cell to retry.', flush=True)
            raise
    receipt = {'archive': str(archive), 'bytes': archive.stat().st_size, 'sha256': sha(archive),
               'included_files': len(sources), 'scope': report['scope'], 'preliminary_development_only': True}
    write_json(destination/'ARCHIVE_RECEIPT.json', receipt)
    print('HANDOFF READY:', archive, flush=True)
    return receipt


def audit_essential_unlocked(root, previous):
    selection = selection_check(root)
    current = [check_run(root, selection, model, parent) for model, parent in PROFILES]
    require(current == previous['runs'] and {k: v for k, v in selection.items() if k not in ('work', 'cfg')} == previous['selection'],
            'Evidence changed since verification; rerun the verification cell in a fresh folder')
    return current


## 2. Set your Sol project and create a separate output folder


In [ ]:
ROOT = Path('/scratch/sshind38/capstone_approach_b')
require(ROOT.is_dir(), 'Open this notebook on Sol; the project folder was not found: ' + str(ROOT))
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
OUT = ROOT / ('method_b_essential_' + STAMP)
OUT.mkdir(exist_ok=False)
print('Existing project:', ROOT)
print('New output folder:', OUT)
print('Scope: mixed 1x, training seed 42, models 0.5B/1.5B, parent seeds 17/123.')


## 3. Verify the four existing runs

Checks data freshness, random priorities, source/length/token controls, saved label checks,
parent/adapter lineage, file hashes, 500 unique question IDs per benchmark, matched evaluator
settings, reported accuracy, and the original 2,000-replicate paired-item bootstrap intervals.
There should be **four VERIFIED messages**. A failed check writes a diagnostic JSON and stops.


In [ ]:
CHECK = audit_essential(ROOT, OUT)
print('\nESSENTIAL EVIDENCE CHECK PASSED: 4 B runs and 8 development comparisons.')
print('Diagnostic/report:', OUT / 'ESSENTIAL_CHECK.json')


## 4. View the seed-42 results

Percentages are development accuracy. Changes and intervals are in percentage points.


In [ ]:
from IPython.display import Markdown, display
lines = ['| Parent profile | Dataset | Parent | B | Change | Paired-item 95% interval |',
         '|---|---|---:|---:|---:|---|']
for item in CHECK['runs']:
    for r in item['rows']:
        lines.append(f"| {r['parent_profile']} | {r['dataset']} | {100*r['parent_accuracy']:.2f}% | "
                     f"{100*r['B_accuracy']:.2f}% | {r['change_percentage_points']:+.2f} | "
                     f"[{r['paired_item_ci95_low_pp']:+.2f}, {r['paired_item_ci95_high_pp']:+.2f}] |")
display(Markdown('\n'.join(lines)))
print('Preliminary development evidence; evaluator protection requires shared verification.')
print('Item intervals concern these fixed checkpoints; they do not estimate training-seed variation.')


## 5. Package the essential handoff

This cell rechecks the evidence and creates a `.tar.gz` in the project root. It includes only
the four selected B adapters, their predictions and training records, relevant parent
predictions, selected-data/bank evidence, and reproducibility code. Original parent weights,
base-model caches, environments, and optimizer-state checkpoints are excluded.

The original results are preserved. Hashing/compression may take several minutes;
progress is printed every 50 files. If interrupted, rerun this cell to retry. If a finished
archive already exists, rerun from Section 2 to create a fresh output folder.


In [ ]:
RECEIPT = package_essential(ROOT, OUT, CHECK)
print('\nSize: %.1f MiB' % (RECEIPT['bytes'] / 1024**2))
print('Archive SHA-256:', RECEIPT['sha256'])
print('Results CSV:', OUT / 'RESULTS.csv')
print('Plain-English handoff:', OUT / 'README.txt')


## 6. Download and report

In **Sol portal → Files**, open `/scratch/sshind38/capstone_approach_b`, click Refresh,
and find the `method_b_essential_...tar.gz` printed above. Select it and choose Download.
The matching folder contains `RESULTS.csv`, `README.txt`, `ESSENTIAL_CHECK.json`, and
`ARCHIVE_RECEIPT.json`. You can upload those small reports here without uploading model weights.

**Completed in this scope:** source/length/budget-controlled fresh-random B at 1×,
training seed 42, both sizes and both original parent seeds; evidence checks and handoff.

**Still separate:** final multi-seed confirmation, compatible A/C comparisons using teammate
predictions, evaluator/protection clearance, protected final evaluation, and any selected
transfer or extra-budget claims. The previous 4× condition was blocked by screened-bank capacity.
Keep negative/inconclusive findings. A successful handoff does not establish that B improves accuracy.
